<a href="https://colab.research.google.com/github/amzad-786githumb/AIR_LLM_Research/blob/main/06_LLM_Recommendation_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# NOTEBOOK 06.1 — ENVIRONMENT AND IMPORTS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.1")
print("CANDIDATE STRATEGY EVALUATION")
print("=" * 100)

import os
import gc
import json
import time
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.linear_model import BayesianRidge, LogisticRegression
from sklearn.ensemble import (
    ExtraTreesRegressor,
    ExtraTreesClassifier,
    RandomForestRegressor,
    RandomForestClassifier,
)

warnings.filterwarnings("ignore")

MASTER_SEED = 42

np.random.seed(MASTER_SEED)

print("NumPy version :", np.__version__)
print("Pandas version:", pd.__version__)
print("Master seed   :", MASTER_SEED)

print("\nEnvironment initialization : PASSED")

NOTEBOOK 06 — PROFILES LOADED
Dataset profile rows : 3
Feature profile rows : 80
Datasets loaded      : 3
Recommendation root  : /content/drive/MyDrive/AIR_LLM_Research/results/recommendations


In [ ]:
# ============================================================
# NOTEBOOK 06.2 — GLOBAL CONFIGURATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.2")
print("GLOBAL EVALUATION CONFIGURATION")
print("=" * 100)


# ------------------------------------------------------------
# Dataset configuration
# ------------------------------------------------------------

if "DATASETS" not in globals():

    DATASETS = [
        "adult_income",
        "bank_marketing",
        "diabetes_130us"
    ]


if "TARGET_REGISTRY" not in globals():

    TARGET_REGISTRY = {
        "adult_income": "income",
        "bank_marketing": "y",
        "diabetes_130us": "readmitted"
    }


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

MASTER_SEED = 42


# ------------------------------------------------------------
# Evaluation sample size
# ------------------------------------------------------------

EVALUATION_SAMPLE_SIZE = 5000


# ------------------------------------------------------------
# Missingness levels
# ------------------------------------------------------------

MISSINGNESS_LEVELS = [
    0.10,
    0.20,
    0.30,
    0.40,
    0.50
]


# ------------------------------------------------------------
# Missingness mechanisms
# ------------------------------------------------------------

MISSINGNESS_MECHANISMS = [
    "MCAR",
    "MAR",
    "MNAR"
]


# ------------------------------------------------------------
# Minimum requirements
# ------------------------------------------------------------

MIN_OBSERVED_VALUES = 30
MIN_EVALUATION_VALUES = 20


# ------------------------------------------------------------
# KNN
# ------------------------------------------------------------

KNN_NEIGHBORS = 5


# ------------------------------------------------------------
# Tree models
# ------------------------------------------------------------

TREE_ESTIMATORS = 100


# ------------------------------------------------------------
# Maximum feature count used by model
# ------------------------------------------------------------

MAX_PREDICTOR_FEATURES = 30


print("\nConfiguration")
print("-" * 100)

print("Datasets                :", DATASETS)
print("Evaluation sample size  :", EVALUATION_SAMPLE_SIZE)
print("Missingness levels      :", MISSINGNESS_LEVELS)
print("Mechanisms              :", MISSINGNESS_MECHANISMS)
print("KNN neighbors           :", KNN_NEIGHBORS)
print("Tree estimators         :", TREE_ESTIMATORS)
print("Master seed             :", MASTER_SEED)

print("\nConfiguration : PASSED")

NOTEBOOK 06 — CANDIDATE STRATEGY REGISTRY
Registry path : /content/drive/MyDrive/AIR_LLM_Research/results/candidates/candidate_strategy_registry.csv
Candidate methods : 9
  - Mean
  - Median
  - Mode
  - KNN
  - MICE
  - MissForest
  - GAIN
  - LLM
  - AutomatedSelection


In [ ]:
# ============================================================
# NOTEBOOK 06.3 — PROJECT DIRECTORIES
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.3")
print("PROJECT DIRECTORIES")
print("=" * 100)


# ------------------------------------------------------------
# Detect project root
# ------------------------------------------------------------

if "PROJECT_ROOT" not in globals():

    possible_roots = [

        Path("/content/drive/MyDrive/AIR-LLM"),

        Path("/content/drive/MyDrive/AIR_LLM"),

        Path("/content/AIR-LLM"),

        Path("/content/AIR_LLM")

    ]

    PROJECT_ROOT = None

    for candidate in possible_roots:

        if candidate.exists():

            PROJECT_ROOT = candidate

            break


    if PROJECT_ROOT is None:

        PROJECT_ROOT = Path(
            "/content/drive/MyDrive/AIR-LLM"
        )


PROJECT_ROOT = Path(PROJECT_ROOT)


# ------------------------------------------------------------
# Output directories
# ------------------------------------------------------------

NOTEBOOK_06_DIR = (
    PROJECT_ROOT
    / "notebook_06_candidate_evaluation"
)

INPUT_DIR = (
    NOTEBOOK_06_DIR
    / "inputs"
)

OUTPUT_DIR = (
    NOTEBOOK_06_DIR
    / "outputs"
)

RESULTS_DIR = (
    OUTPUT_DIR
    / "results"
)

REPORT_DIR = (
    OUTPUT_DIR
    / "reports"
)

LOG_DIR = (
    OUTPUT_DIR
    / "logs"
)

MANIFEST_DIR = (
    OUTPUT_DIR
    / "manifest"
)


for directory in [
    NOTEBOOK_06_DIR,
    INPUT_DIR,
    OUTPUT_DIR,
    RESULTS_DIR,
    REPORT_DIR,
    LOG_DIR,
    MANIFEST_DIR
]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


print(
    f"Project root : {PROJECT_ROOT}"
)

print(
    f"Notebook 06  : {NOTEBOOK_06_DIR}"
)

print("\nDirectories : READY")

STRUCTURED LLM INPUT CONSTRUCTED
Incomplete features : 9


,dataset_id,feature,missing_rate
0,diabetes_130us,race,0.022601
1,diabetes_130us,weight,0.968801
2,diabetes_130us,payer_code,0.396633
3,diabetes_130us,medical_specialty,0.490395
4,diabetes_130us,diag_1,0.000213
5,diabetes_130us,diag_2,0.003488
6,diabetes_130us,diag_3,0.014036
7,diabetes_130us,max_glu_serum,0.947657
8,diabetes_130us,A1Cresult,0.832998


In [ ]:
# ============================================================
# NOTEBOOK 06.4 — PREVIOUS NOTEBOOK ARTIFACT DISCOVERY
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.4")
print("PREVIOUS ARTIFACT DISCOVERY")
print("=" * 100)


def find_artifact(filename):

    search_roots = [

        PROJECT_ROOT,

        PROJECT_ROOT / "notebook_04_dataset_feature_profiling",

        PROJECT_ROOT / "notebook_05_local_llm_recommendation",

        PROJECT_ROOT / "outputs",

        PROJECT_ROOT / "data"

    ]

    for root in search_roots:

        candidate = root / filename

        if candidate.exists():

            return candidate


    # Recursive fallback

    matches = list(
        PROJECT_ROOT.rglob(filename)
    )

    if matches:

        return matches[0]

    return None


ARTIFACT_NAMES = {

    "feature_profile":
        "feature_profile.csv",

    "dataset_profile":
        "dataset_profile.csv",

    "missingness_profile":
        "missingness_profile.csv",

    "numeric_dependencies":
        "numeric_dependencies.csv",

    "categorical_associations":
        "categorical_associations.csv",

    "mutual_information":
        "mutual_information.csv",

    "task_relevance":
        "task_relevance.csv",

    "llm_recommendations":
        "recommendations.csv",

    "llm_raw_responses":
        "llm_raw_responses.csv"

}


ARTIFACT_PATHS = {}

for key, filename in ARTIFACT_NAMES.items():

    path = find_artifact(filename)

    ARTIFACT_PATHS[key] = path

    print(
        f"{key:30s}: "
        f"{path if path else 'NOT FOUND'}"
    )


print("\nArtifact discovery completed.")

LLM system prompt constructed.


In [ ]:
# ============================================================
# NOTEBOOK 06.5 — LOAD PROFILING ARTIFACTS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.5")
print("LOAD NOTEBOOK 04 PROFILING ARTIFACTS")
print("=" * 100)


def load_csv_artifact(
    key,
    required=True
):

    path = ARTIFACT_PATHS.get(key)

    if path is None:

        if required:

            raise FileNotFoundError(
                f"Required artifact not found: {key}"
            )

        return pd.DataFrame()

    df = pd.read_csv(path)

    print(
        f"{key:30s}: "
        f"{df.shape[0]:,} rows × "
        f"{df.shape[1]:,} columns"
    )

    return df


FEATURE_PROFILE_DF = load_csv_artifact(
    "feature_profile",
    required=True
)

DATASET_PROFILE_DF = load_csv_artifact(
    "dataset_profile",
    required=False
)

MISSINGNESS_PROFILE_DF = load_csv_artifact(
    "missingness_profile",
    required=False
)

NUMERIC_DEPENDENCY_DF = load_csv_artifact(
    "numeric_dependencies",
    required=False
)

CATEGORICAL_ASSOCIATION_DF = load_csv_artifact(
    "categorical_associations",
    required=False
)

MUTUAL_INFORMATION_DF = load_csv_artifact(
    "mutual_information",
    required=False
)

TASK_RELEVANCE_DF = load_csv_artifact(
    "task_relevance",
    required=False
)

print("\nProfiling artifacts : LOADED")

Feature-level prompts constructed: 9


In [ ]:
# ============================================================
# NOTEBOOK 06.6 — LOAD LLM RECOMMENDATIONS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.6")
print("LOAD LOCAL LLM RECOMMENDATIONS")
print("=" * 100)


LLM_RECOMMENDATION_CANDIDATES = [

    "recommendations.csv",

    "llm_recommendations.csv",

    "strategy_recommendations.csv",

    "candidate_recommendations.csv",

    "final_recommendations.csv"

]


LLM_RECOMMENDATION_PATH = None


for filename in LLM_RECOMMENDATION_CANDIDATES:

    path = find_artifact(filename)

    if path is not None:

        LLM_RECOMMENDATION_PATH = path

        break


if LLM_RECOMMENDATION_PATH is not None:

    LLM_RECOMMENDATION_DF = pd.read_csv(
        LLM_RECOMMENDATION_PATH
    )

    print(
        "Recommendation file:",
        LLM_RECOMMENDATION_PATH
    )

    print(
        "Rows:",
        len(LLM_RECOMMENDATION_DF)
    )

    print(
        "Columns:",
        list(
            LLM_RECOMMENDATION_DF.columns
        )
    )

else:

    print(
        "WARNING: LLM recommendation "
        "artifact was not found."
    )

    LLM_RECOMMENDATION_DF = pd.DataFrame()


print("\nLLM recommendation loading : COMPLETED")

NOTEBOOK 06 — LLM CONFIGURATION
API key status : AVAILABLE
LLM model      : gpt-4o-mini
Temperature    : 0.0
Max output     : 1000


In [ ]:
# ============================================================
# NOTEBOOK 06.7 — LOAD TRAINING DATA
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.7")
print("LOAD TRAINING DATA")
print("=" * 100)


if "TRAINING_DATA" not in globals():

    TRAINING_DATA = {}


def find_processed_dataset(dataset_id):

    candidate_names = [

        f"{dataset_id}_processed.csv",

        f"{dataset_id}.csv"

    ]

    search_roots = [

        PROJECT_ROOT,

        PROJECT_ROOT / "data",

        PROJECT_ROOT / "processed_data",

        PROJECT_ROOT / "datasets",

        PROJECT_ROOT / "notebook_02_data_preprocessing"

    ]

    for root in search_roots:

        for filename in candidate_names:

            path = root / filename

            if path.exists():

                return path


    for filename in candidate_names:

        matches = list(
            PROJECT_ROOT.rglob(filename)
        )

        if matches:

            return matches[0]

    return None


for dataset_id in DATASETS:

    if dataset_id in TRAINING_DATA:

        print(
            f"{dataset_id:25s}: "
            f"already loaded"
        )

        continue


    path = find_processed_dataset(
        dataset_id
    )

    if path is None:

        raise FileNotFoundError(
            f"Training dataset not found: "
            f"{dataset_id}"
        )


    TRAINING_DATA[dataset_id] = pd.read_csv(
        path
    )

    print(
        f"{dataset_id:25s}: "
        f"{TRAINING_DATA[dataset_id].shape}"
    )


print("\nTraining datasets : READY")

NOTEBOOK 06 — LLM CONFIGURATION
API key status : AVAILABLE
LLM model      : gpt-4o-mini
Temperature    : 0.0
Max tokens     : 1000
Max retries    : 5
Request delay  : 1.0s

GENERATING AIR-LLM CANDIDATE RECOMMENDATIONS
Total feature prompts: 9

[1/9] diabetes_130us → race
      Status: FAILED
      Error type: quota_error
      Error: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

Account/API configuration problem detected.
Stopping further API requests to avoid unnecessary calls.

LLM RECOMMENDATION GENERATION COMPLETE
Total requests       : 9
Successful            : 0
Failed                : 1
Not attempted         : 8

ERROR SUMMARY
------------------------------------------------------------------------------------------


,error_type,count
0,quota_error,9



FAILED REQUESTS
------------------------------------------------------------------------------------------


,dataset_id,feature,error_type,attempts,error
0,diabetes_130us,race,quota_error,1,Error code: 429 - {'error': {'message': 'You e...



Raw recommendation results saved to:
air_llm_raw_recommendations.csv

⚠ SOME AIR-LLM REQUESTS FAILED.
Review error_type and error columns before continuing.


In [ ]:
# ============================================================
# NOTEBOOK 06.8 — CANDIDATE STRATEGY REGISTRY
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.8")
print("CANDIDATE IMPUTATION STRATEGY REGISTRY")
print("=" * 100)


STRATEGY_REGISTRY = {

    "mean": {
        "name":
            "Mean Imputation",
        "family":
            "statistical",
        "supported_types":
            ["numeric"],
        "baseline":
            True
    },

    "median": {
        "name":
            "Median Imputation",
        "family":
            "statistical",
        "supported_types":
            ["numeric"],
        "baseline":
            True
    },

    "mode": {
        "name":
            "Mode Imputation",
        "family":
            "statistical",
        "supported_types":
            ["categorical"],
        "baseline":
            True
    },

    "knn": {
        "name":
            "KNN Imputation",
        "family":
            "distance_based",
        "supported_types":
            ["numeric", "categorical"],
        "baseline":
            False
    },

    "mice_bayesian": {
        "name":
            "MICE Bayesian Ridge",
        "family":
            "iterative",
        "supported_types":
            ["numeric"],
        "baseline":
            False
    },

    "extra_trees": {
        "name":
            "Extra Trees Imputation",
        "family":
            "ensemble",
        "supported_types":
            ["numeric", "categorical"],
        "baseline":
            False
    },

    "random_forest": {
        "name":
            "Random Forest Imputation",
        "family":
            "ensemble",
        "supported_types":
            ["numeric", "categorical"],
        "baseline":
            False
    }

}


STRATEGY_REGISTRY_DF = pd.DataFrame(
    [
        {
            "strategy_id": strategy_id,
            **metadata
        }
        for strategy_id, metadata
        in STRATEGY_REGISTRY.items()
    ]
)


display(
    STRATEGY_REGISTRY_DF
)

print(
    f"\nRegistered strategies: "
    f"{len(STRATEGY_REGISTRY)}"
)

NOTEBOOK 06.7 — PARSE STRUCTURED LLM RESPONSES

Total responses       : 9
Successfully parsed   : 0
API failures          : 9
Invalid JSON          : 0
Missing fields        : 0

PARSE STATUS DISTRIBUTION


,parse_status,count
0,not_parsed,9



API ERROR DISTRIBUTION


,dataset_id,feature,api_status,api_error_type,parse_status,parse_error
0,diabetes_130us,None,failed,quota_error,not_parsed,Error code: 429 - {'error': {'message': 'You e...
1,diabetes_130us,None,not_attempted,quota_error,not_parsed,Skipped because account/API configuration prob...
2,diabetes_130us,None,not_attempted,quota_error,not_parsed,Skipped because account/API configuration prob...
3,diabetes_130us,None,not_attempted,quota_error,not_parsed,Skipped because account/API configuration prob...
4,diabetes_130us,None,not_attempted,quota_error,not_parsed,Skipped because account/API configuration prob...
5,diabetes_130us,None,not_attempted,quota_error,not_parsed,Skipped because account/API configuration prob...
6,diabetes_130us,None,not_attempted,quota_error,not_parsed,Skipped because account/API configuration prob...
7,diabetes_130us,None,not_attempted,quota_error,not_parsed,Skipped because account/API configuration prob...
8,diabetes_130us,None,not_attempted,quota_error,not_parsed,Skipped because account/API configuration prob...



Saved parsed results to: air_llm_parsed_recommendations.csv

NOTEBOOK 06.7 PARSING COMPLETE


In [ ]:
# ============================================================
# NOTEBOOK 06.9 — STRATEGY NORMALIZATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.9")
print("STRATEGY ID NORMALIZATION")
print("=" * 100)


STRATEGY_ALIASES = {

    "mean":
        "mean",

    "mean_imputation":
        "mean",

    "mean imputation":
        "mean",

    "median":
        "median",

    "median_imputation":
        "median",

    "median imputation":
        "median",

    "mode":
        "mode",

    "mode_imputation":
        "mode",

    "mode imputation":
        "mode",

    "knn":
        "knn",

    "knn_imputer":
        "knn",

    "knn imputation":
        "knn",

    "mice":
        "mice_bayesian",

    "mice_bayesian":
        "mice_bayesian",

    "iterative":
        "mice_bayesian",

    "iterative_imputer":
        "mice_bayesian",

    "bayesian_ridge":
        "mice_bayesian",

    "extra_trees":
        "extra_trees",

    "extra trees":
        "extra_trees",

    "extratrees":
        "extra_trees",

    "missforest":
        "extra_trees",

    "miss_forest":
        "extra_trees",

    "random_forest":
        "random_forest",

    "random forest":
        "random_forest",

    "rf":
        "random_forest"

}


def normalize_strategy_id(strategy):

    if pd.isna(strategy):

        return None

    value = str(
        strategy
    ).strip().lower()

    value = value.replace(
        "-",
        "_"
    )

    value = " ".join(
        value.split()
    )

    if value in STRATEGY_ALIASES:

        return STRATEGY_ALIASES[value]

    return None


print(
    "Strategy normalization : READY"
)

RECOMMENDATION FORMAT VALIDATION


,dataset_id,feature,valid,errors
0,diabetes_130us,race,False,response_not_valid_json
1,diabetes_130us,weight,False,response_not_valid_json
2,diabetes_130us,payer_code,False,response_not_valid_json
3,diabetes_130us,medical_specialty,False,response_not_valid_json
4,diabetes_130us,diag_1,False,response_not_valid_json
5,diabetes_130us,diag_2,False,response_not_valid_json
6,diabetes_130us,diag_3,False,response_not_valid_json
7,diabetes_130us,max_glu_serum,False,response_not_valid_json
8,diabetes_130us,A1Cresult,False,response_not_valid_json


In [ ]:
# ============================================================
# NOTEBOOK 06.10 — FEATURE TYPE DETECTION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.10")
print("FEATURE TYPE DETECTION")
print("=" * 100)


def detect_feature_type(
    series
):

    if pd.api.types.is_numeric_dtype(
        series
    ):

        return "numeric"

    return "categorical"


def get_feature_type(
    df,
    feature
):

    return detect_feature_type(
        df[feature]
    )


print(
    "Feature-type detection : READY"
)

RANKED TOP-K RECOMMENDATIONS


KeyError: "None of [Index(['dataset_id', 'feature'], dtype='object')] are in the [columns]"

In [ ]:
# ============================================================
# NOTEBOOK 06.11 — DETERMINISTIC EVALUATION SAMPLING
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.11")
print("DETERMINISTIC EVALUATION SAMPLING")
print("=" * 100)


EVALUATION_DATA = {}


for dataset_id in DATASETS:

    df = TRAINING_DATA[dataset_id].copy()

    if len(df) > EVALUATION_SAMPLE_SIZE:

        sample_df = df.sample(
            EVALUATION_SAMPLE_SIZE,
            random_state=MASTER_SEED
        ).reset_index(
            drop=True
        )

    else:

        sample_df = df.reset_index(
            drop=True
        )


    EVALUATION_DATA[
        dataset_id
    ] = sample_df


    print(
        f"{dataset_id:25s}: "
        f"{len(sample_df):,} rows"
    )


print("\nEvaluation datasets : READY")

NOTEBOOK 06.10 — RECORD LLM RATIONALE

Total features        : 9
Rationales available  : 0
Missing/empty         : 0
API failures          : 9
Not parsed            : 0

RATIONALE STATUS


,rationale_status,count
0,api_failure,9



Rationale records saved to:
air_llm_rationale.csv

NOTEBOOK 06.10 COMPLETE


In [ ]:
# ============================================================
# NOTEBOOK 06.12 — MISSINGNESS MASK GENERATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.12")
print("MCAR / MAR / MNAR MASK GENERATION")
print("=" * 100)


def generate_mcar_mask(
    series,
    missing_rate,
    seed
):

    rng = np.random.default_rng(
        seed
    )

    observed = (
        series.notna()
    )

    eligible = np.where(
        observed.to_numpy()
    )[0]

    n_missing = int(
        np.floor(
            len(eligible)
            * missing_rate
        )
    )

    n_missing = min(
        n_missing,
        len(eligible)
    )

    selected = rng.choice(
        eligible,
        size=n_missing,
        replace=False
    )

    mask = np.zeros(
        len(series),
        dtype=bool
    )

    mask[selected] = True

    return mask


def generate_mar_mask(
    df,
    target,
    missing_rate,
    seed
):

    rng = np.random.default_rng(
        seed
    )

    observed = df[target].notna()

    eligible = np.where(
        observed.to_numpy()
    )[0]

    if len(eligible) == 0:

        return np.zeros(
            len(df),
            dtype=bool
        )


    predictors = [
        c
        for c in df.columns
        if c != target
    ]

    if not predictors:

        return generate_mcar_mask(
            df[target],
            missing_rate,
            seed
        )


    # Select the first usable predictor
    predictor = None

    for c in predictors:

        if df[c].notna().sum() >= 30:

            predictor = c

            break


    if predictor is None:

        return generate_mcar_mask(
            df[target],
            missing_rate,
            seed
        )


    values = df[predictor].copy()

    if pd.api.types.is_numeric_dtype(
        values
    ):

        score = values.fillna(
            values.median()
        ).to_numpy(
            dtype=float
        )

    else:

        codes = pd.factorize(
            values.fillna("__MISSING__")
            .astype(str)
        )[0]

        score = codes.astype(float)


    score = np.nan_to_num(
        score
    )

    order = np.argsort(
        score
    )

    # Bias missingness toward higher-risk region
    ordered_eligible = [
        idx
        for idx in order
        if idx in set(eligible)
    ]

    n_missing = int(
        np.floor(
            len(eligible)
            * missing_rate
        )
    )

    # Probabilistic selection from upper half
    upper_start = max(
        0,
        len(ordered_eligible)
        // 2
    )

    preferred = ordered_eligible[
        upper_start:
    ]

    if len(preferred) < n_missing:

        preferred = ordered_eligible


    if len(preferred) >= n_missing:

        selected = rng.choice(
            preferred,
            size=n_missing,
            replace=False
        )

    else:

        selected = rng.choice(
            eligible,
            size=n_missing,
            replace=False
        )


    mask = np.zeros(
        len(df),
        dtype=bool
    )

    mask[selected] = True

    return mask


def generate_mnar_mask(
    series,
    missing_rate,
    seed
):

    rng = np.random.default_rng(
        seed
    )

    observed = series.notna()

    eligible = np.where(
        observed.to_numpy()
    )[0]

    if len(eligible) == 0:

        return np.zeros(
            len(series),
            dtype=bool
        )


    values = series.copy()

    if pd.api.types.is_numeric_dtype(
        values
    ):

        numeric = values.fillna(
            values.median()
        ).to_numpy(
            dtype=float
        )

        order = np.argsort(
            numeric
        )

    else:

        codes = pd.factorize(
            values.fillna("__MISSING__")
            .astype(str)
        )[0]

        order = np.argsort(
            codes
        )


    # Concentrate missingness in high-value region
    eligible_ordered = [
        idx
        for idx in order[::-1]
        if idx in set(eligible)
    ]

    n_missing = int(
        np.floor(
            len(eligible)
            * missing_rate
        )
    )

    n_missing = min(
        n_missing,
        len(eligible_ordered)
    )

    selected = eligible_ordered[
        :n_missing
    ]

    mask = np.zeros(
        len(series),
        dtype=bool
    )

    mask[selected] = True

    return mask


print(
    "Missingness generators : READY"
)

NOTEBOOK 06.11 — RECORD LLM CONFIDENCE

Total features       : 9
Valid confidence     : 0
Invalid confidence   : 0
API failures         : 9
Not parsed           : 0

CONFIDENCE STATUS


,confidence_status,count
0,api_failure,9



CONFIDENCE LEVEL DISTRIBUTION


,confidence_level,count
0,unavailable,9



Confidence records saved to:
air_llm_confidence.csv

NOTEBOOK 06.11 COMPLETE


In [ ]:
# ============================================================
# NOTEBOOK 06.13 — UNIFIED MISSINGNESS GENERATOR
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.13")
print("UNIFIED MISSINGNESS GENERATOR")
print("=" * 100)


def generate_missingness_mask(
    df,
    target,
    mechanism,
    missing_rate,
    seed
):

    if mechanism == "MCAR":

        return generate_mcar_mask(
            df[target],
            missing_rate,
            seed
        )


    if mechanism == "MAR":

        return generate_mar_mask(
            df,
            target,
            missing_rate,
            seed
        )


    if mechanism == "MNAR":

        return generate_mnar_mask(
            df[target],
            missing_rate,
            seed
        )


    raise ValueError(
        f"Unsupported mechanism: "
        f"{mechanism}"
    )


print(
    "Unified missingness generator : READY"
)

NOTEBOOK 06.12 — SAVE AIR-LLM RECOMMENDATIONS

AIR-LLM RECOMMENDATION SUMMARY
Total feature records : 9
Complete records      : 0
Incomplete records    : 9

RECOMMENDATION STATUS


,recommendation_status,count
0,api_failure,9



This is expected while the OpenAI API quota is exhausted.

FILES SAVED
CSV              : air_llm_recommendations.csv
Excel            : air_llm_recommendations.xlsx
Valid-only CSV   : air_llm_valid_recommendations.csv

NOTEBOOK 06.12 COMPLETE


In [ ]:
# ============================================================
# NOTEBOOK 06.14 — PREDICTOR MATRIX CONSTRUCTION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.14")
print("PREDICTOR MATRIX CONSTRUCTION")
print("=" * 100)


def select_predictors(
    df,
    target
):

    predictors = [
        c
        for c in df.columns
        if c != target
    ]


    usable = []

    for feature in predictors:

        non_missing = (
            df[feature]
            .notna()
            .sum()
        )

        if non_missing >= MIN_OBSERVED_VALUES:

            usable.append(feature)


    if len(usable) > MAX_PREDICTOR_FEATURES:

        numeric = [
            c
            for c in usable
            if pd.api.types.is_numeric_dtype(
                df[c]
            )
        ]

        categorical = [
            c
            for c in usable
            if c not in numeric
        ]

        usable = (
            numeric[:MAX_PREDICTOR_FEATURES]
            +
            categorical[:MAX_PREDICTOR_FEATURES]
        )

        usable = usable[
            :MAX_PREDICTOR_FEATURES
        ]


    return usable


print(
    "Predictor construction : READY"
)

In [ ]:
# ============================================================
# NOTEBOOK 06.15 — SAFE FEATURE ENCODING
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.15")
print("SAFE FEATURE ENCODING")
print("=" * 100)


def encode_predictors(
    X_train,
    X_test
):

    X_train = X_train.copy()
    X_test = X_test.copy()


    numeric_cols = [
        c
        for c in X_train.columns
        if pd.api.types.is_numeric_dtype(
            X_train[c]
        )
    ]

    categorical_cols = [
        c
        for c in X_train.columns
        if c not in numeric_cols
    ]


    # Numeric imputation
    for c in numeric_cols:

        median = X_train[c].median()

        X_train[c] = X_train[c].fillna(
            median
        )

        X_test[c] = X_test[c].fillna(
            median
        )


    # Categorical encoding
    if categorical_cols:

        encoder = OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        )

        train_values = (
            X_train[categorical_cols]
            .fillna("__MISSING__")
            .astype(str)
        )

        test_values = (
            X_test[categorical_cols]
            .fillna("__MISSING__")
            .astype(str)
        )

        X_train[categorical_cols] = (
            encoder.fit_transform(
                train_values
            )
        )

        X_test[categorical_cols] = (
            encoder.transform(
                test_values
            )
        )


    return (
        X_train.astype(float),
        X_test.astype(float)
    )


print(
    "Encoding system : READY"
)

In [ ]:
# ============================================================
# NOTEBOOK 06.16 — CANDIDATE IMPUTATION FUNCTIONS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.16")
print("CANDIDATE IMPUTATION FUNCTIONS")
print("=" * 100)


def impute_mean(
    train_y,
    missing_y
):

    value = train_y.mean()

    return np.full(
        len(missing_y),
        value
    )


def impute_median(
    train_y,
    missing_y
):

    value = train_y.median()

    return np.full(
        len(missing_y),
        value
    )


def impute_mode(
    train_y,
    missing_y
):

    mode = train_y.mode()

    if len(mode) == 0:

        value = train_y.iloc[0]

    else:

        value = mode.iloc[0]

    return np.repeat(
        value,
        len(missing_y)
    )


def fit_knn(
    feature_type
):

    if feature_type == "numeric":

        return KNeighborsRegressor(
            n_neighbors=KNN_NEIGHBORS,
            weights="distance"
        )

    return KNeighborsClassifier(
        n_neighbors=KNN_NEIGHBORS,
        weights="distance"
    )


def fit_mice_bayesian():

    return BayesianRidge()


def fit_extra_trees(
    feature_type
):

    if feature_type == "numeric":

        return ExtraTreesRegressor(
            n_estimators=TREE_ESTIMATORS,
            random_state=MASTER_SEED,
            n_jobs=-1,
            min_samples_leaf=2
        )

    return ExtraTreesClassifier(
        n_estimators=TREE_ESTIMATORS,
        random_state=MASTER_SEED,
        n_jobs=-1,
        min_samples_leaf=2
    )


def fit_random_forest(
    feature_type
):

    if feature_type == "numeric":

        return RandomForestRegressor(
            n_estimators=TREE_ESTIMATORS,
            random_state=MASTER_SEED,
            n_jobs=-1,
            min_samples_leaf=2
        )

    return RandomForestClassifier(
        n_estimators=TREE_ESTIMATORS,
        random_state=MASTER_SEED,
        n_jobs=-1,
        min_samples_leaf=2
    )


print(
    "Candidate estimators : READY"
)

In [ ]:
# ============================================================
# NOTEBOOK 06.17 — CANDIDATE EVALUATION ENGINE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.17")
print("CANDIDATE EVALUATION ENGINE")
print("=" * 100)


def evaluate_single_candidate(
    df,
    target,
    strategy_id,
    mechanism,
    missing_rate,
    seed
):

    feature_type = detect_feature_type(
        df[target]
    )

    predictors = select_predictors(
        df,
        target
    )


    if len(predictors) == 0:

        return {
            "status": "SKIPPED",
            "error":
                "No usable predictors"
        }


    mask = generate_missingness_mask(
        df,
        target,
        mechanism,
        missing_rate,
        seed
    )


    observed_mask = (
        df[target].notna()
    )

    evaluation_mask = (
        mask
        & observed_mask
    )


    n_evaluation = int(
        evaluation_mask.sum()
    )


    if n_evaluation < MIN_EVALUATION_VALUES:

        return {
            "status": "SKIPPED",
            "error":
                "Insufficient evaluation values",
            "n_evaluation":
                n_evaluation
        }


    working_df = df.copy()

    true_values = (
        working_df.loc[
            evaluation_mask,
            target
        ].copy()
    )


    # --------------------------------------------------------
    # Remove target values to simulate missingness
    # --------------------------------------------------------

    working_df.loc[
        evaluation_mask,
        target
    ] = np.nan


    train_mask = (
        ~evaluation_mask
        & working_df[target].notna()
    )


    test_mask = evaluation_mask


    X_train = (
        working_df.loc[
            train_mask,
            predictors
        ]
    )

    X_test = (
        working_df.loc[
            test_mask,
            predictors
        ]
    )

    y_train = (
        working_df.loc[
            train_mask,
            target
        ]
    )


    if len(y_train) < MIN_OBSERVED_VALUES:

        return {
            "status": "SKIPPED",
            "error":
                "Insufficient training observations"
        }


    start_time = time.perf_counter()


    try:

        # ----------------------------------------------------
        # Statistical baselines
        # ----------------------------------------------------

        if strategy_id == "mean":

            predictions = impute_mean(
                y_train,
                true_values
            )


        elif strategy_id == "median":

            predictions = impute_median(
                y_train,
                true_values
            )


        elif strategy_id == "mode":

            predictions = impute_mode(
                y_train,
                true_values
            )


        # ----------------------------------------------------
        # Model-based methods
        # ----------------------------------------------------

        else:

            X_train_encoded, X_test_encoded = (
                encode_predictors(
                    X_train,
                    X_test
                )
            )


            if strategy_id == "knn":

                model = fit_knn(
                    feature_type
                )


            elif strategy_id == "mice_bayesian":

                if feature_type != "numeric":

                    return {
                        "status": "SKIPPED",
                        "error":
                            "MICE Bayesian candidate "
                            "currently supports numeric targets only"
                    }

                model = fit_mice_bayesian()


            elif strategy_id == "extra_trees":

                model = fit_extra_trees(
                    feature_type
                )


            elif strategy_id == "random_forest":

                model = fit_random_forest(
                    feature_type
                )


            else:

                return {
                    "status": "SKIPPED",
                    "error":
                        f"Unknown strategy: "
                        f"{strategy_id}"
                }


            model.fit(
                X_train_encoded,
                y_train
            )

            predictions = model.predict(
                X_test_encoded
            )


    except Exception as e:

        return {
            "status": "FAILED",
            "error":
                str(e)
        }


    runtime_seconds = (
        time.perf_counter()
        - start_time
    )


    predictions = np.asarray(
        predictions
    )


    # --------------------------------------------------------
    # Numeric metrics
    # --------------------------------------------------------

    if feature_type == "numeric":

        true_numeric = np.asarray(
            true_values,
            dtype=float
        )

        pred_numeric = np.asarray(
            predictions,
            dtype=float
        )

        rmse = np.sqrt(
            mean_squared_error(
                true_numeric,
                pred_numeric
            )
        )

        mae = mean_absolute_error(
            true_numeric,
            pred_numeric
        )

        try:

            r2 = r2_score(
                true_numeric,
                pred_numeric
            )

        except Exception:

            r2 = np.nan


        scale = np.std(
            true_numeric
        )

        nrmse = (
            rmse / scale
            if scale > 0
            else np.nan
        )


        return {

            "status":
                "SUCCESS",

            "feature_type":
                "numeric",

            "n_evaluation":
                n_evaluation,

            "rmse":
                float(rmse),

            "mae":
                float(mae),

            "nrmse":
                float(nrmse)
                if pd.notna(nrmse)
                else np.nan,

            "r2":
                float(r2)
                if pd.notna(r2)
                else np.nan,

            "accuracy":
                np.nan,

            "balanced_accuracy":
                np.nan,

            "macro_f1":
                np.nan,

            "runtime_seconds":
                float(runtime_seconds)

        }


    # --------------------------------------------------------
    # Categorical metrics
    # --------------------------------------------------------

    true_cat = (
        pd.Series(
            true_values
        )
        .astype(str)
    )

    pred_cat = (
        pd.Series(
            predictions
        )
        .astype(str)
    )


    accuracy = accuracy_score(
        true_cat,
        pred_cat
    )

    balanced_accuracy = (
        balanced_accuracy_score(
            true_cat,
            pred_cat
        )
    )

    macro_f1 = f1_score(
        true_cat,
        pred_cat,
        average="macro",
        zero_division=0
    )


    return {

        "status":
            "SUCCESS",

        "feature_type":
            "categorical",

        "n_evaluation":
            n_evaluation,

        "rmse":
            np.nan,

        "mae":
            np.nan,

        "nrmse":
            np.nan,

        "r2":
            np.nan,

        "accuracy":
            float(accuracy),

        "balanced_accuracy":
            float(
                balanced_accuracy
            ),

        "macro_f1":
            float(
                macro_f1
            ),

        "runtime_seconds":
            float(runtime_seconds)

    }


print(
    "Evaluation engine : READY"
)

In [ ]:
# ============================================================
# NOTEBOOK 06.18 — EXTRACT LLM CANDIDATES
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.18")
print("EXTRACT LLM-RECOMMENDED CANDIDATES")
print("=" * 100)


def extract_llm_candidates(
    dataset_id,
    feature
):

    if LLM_RECOMMENDATION_DF.empty:

        return []


    df = LLM_RECOMMENDATION_DF.copy()


    # --------------------------------------------------------
    # Identify dataset and feature columns
    # --------------------------------------------------------

    dataset_col = None

    for c in [
        "dataset_id",
        "dataset"
    ]:

        if c in df.columns:

            dataset_col = c

            break


    feature_col = None

    for c in [
        "feature",
        "feature_name"
    ]:

        if c in df.columns:

            feature_col = c

            break


    if (
        dataset_col is None
        or
        feature_col is None
    ):

        return []


    subset = df[
        (
            df[dataset_col].astype(str)
            == str(dataset_id)
        )
        &
        (
            df[feature_col].astype(str)
            == str(feature)
        )
    ]


    candidate_columns = [
        "strategy_id",
        "strategy",
        "recommended_strategy",
        "candidate_strategy"
    ]


    strategy_col = None

    for c in candidate_columns:

        if c in subset.columns:

            strategy_col = c

            break


    if strategy_col is None:

        return []


    candidates = []

    for value in subset[
        strategy_col
    ].tolist():

        normalized = normalize_strategy_id(
            value
        )

        if normalized is not None:

            candidates.append(
                normalized
            )


    return list(
        dict.fromkeys(
            candidates
        )
    )


print(
    "LLM candidate extraction : READY"
)

In [ ]:
# ============================================================
# NOTEBOOK 06.19 — CANDIDATE SELECTION POLICY
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.19")
print("CANDIDATE SELECTION POLICY")
print("=" * 100)


BASELINE_STRATEGIES = [
    "mean",
    "median",
    "mode"
]


def get_candidate_strategies(
    feature_type,
    llm_candidates
):

    candidates = []

    # --------------------------------------------------------
    # Always include applicable baselines
    # --------------------------------------------------------

    for strategy in BASELINE_STRATEGIES:

        if feature_type in STRATEGY_REGISTRY[
            strategy
        ]["supported_types"]:

            candidates.append(
                strategy
            )


    # --------------------------------------------------------
    # Add LLM candidates
    # --------------------------------------------------------

    for strategy in llm_candidates:

        if strategy not in STRATEGY_REGISTRY:

            continue

        if (
            feature_type
            in
            STRATEGY_REGISTRY[
                strategy
            ]["supported_types"]
        ):

            candidates.append(
                strategy
            )


    # --------------------------------------------------------
    # Add model-based candidates for
    # empirical benchmark completeness
    # --------------------------------------------------------

    for strategy in [
        "knn",
        "mice_bayesian",
        "extra_trees",
        "random_forest"
    ]:

        if strategy not in STRATEGY_REGISTRY:

            continue

        if (
            feature_type
            in
            STRATEGY_REGISTRY[
                strategy
            ]["supported_types"]
        ):

            candidates.append(
                strategy
            )


    return list(
        dict.fromkeys(
            candidates
        )
    )


print(
    "Candidate selection policy : READY"
)

In [ ]:
# ============================================================
# NOTEBOOK 06.20 — EVALUATION PLAN
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.20")
print("BUILD EVALUATION PLAN")
print("=" * 100)


EVALUATION_PLAN_ROWS = []


for dataset_id in DATASETS:

    df = EVALUATION_DATA[
        dataset_id
    ]

    target = TARGET_REGISTRY[
        dataset_id
    ]


    if target not in df.columns:

        print(
            f"WARNING: target missing "
            f"for {dataset_id}"
        )

        continue


    features = [
        c
        for c in df.columns
        if c != target
    ]


    for feature in features:

        feature_type = detect_feature_type(
            df[feature]
        )


        llm_candidates = (
            extract_llm_candidates(
                dataset_id,
                feature
            )
        )


        candidates = get_candidate_strategies(
            feature_type,
            llm_candidates
        )


        for mechanism in (
            MISSINGNESS_MECHANISMS
        ):

            for missing_rate in (
                MISSINGNESS_LEVELS
            ):

                for strategy_id in candidates:

                    EVALUATION_PLAN_ROWS.append({

                        "dataset_id":
                            dataset_id,

                        "feature":
                            feature,

                        "target":
                            target,

                        "feature_type":
                            feature_type,

                        "strategy_id":
                            strategy_id,

                        "llm_recommended":
                            strategy_id
                            in
                            llm_candidates,

                        "missingness_mechanism":
                            mechanism,

                        "missingness_rate":
                            missing_rate,

                        "seed":
                            MASTER_SEED

                    })


EVALUATION_PLAN_DF = pd.DataFrame(
    EVALUATION_PLAN_ROWS
)


print(
    f"Evaluation configurations: "
    f"{len(EVALUATION_PLAN_DF):,}"
)

display(
    EVALUATION_PLAN_DF.head(20)
)

In [ ]:
# ============================================================
# NOTEBOOK 06.21 — RUN CANDIDATE EVALUATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.21")
print("RUN CANDIDATE STRATEGY EVALUATION")
print("=" * 100)


EVALUATION_RESULTS = []


total = len(
    EVALUATION_PLAN_DF
)


for index, row in (
    EVALUATION_PLAN_DF
    .iterrows()
):

    dataset_id = row[
        "dataset_id"
    ]

    feature = row[
        "feature"
    ]

    mechanism = row[
        "missingness_mechanism"
    ]

    missing_rate = float(
        row[
            "missingness_rate"
        ]
    )

    strategy_id = row[
        "strategy_id"
    ]

    df = EVALUATION_DATA[
        dataset_id
    ]


    result = evaluate_single_candidate(

        df=df,

        target=feature,

        strategy_id=strategy_id,

        mechanism=mechanism,

        missing_rate=missing_rate,

        seed=MASTER_SEED
        + index

    )


    result.update({

        "dataset_id":
            dataset_id,

        "feature":
            feature,

        "strategy_id":
            strategy_id,

        "missingness_mechanism":
            mechanism,

        "missingness_rate":
            missing_rate,

        "llm_recommended":
            bool(
                row[
                    "llm_recommended"
                ]
            ),

        "seed":
            MASTER_SEED
            + index

    })


    EVALUATION_RESULTS.append(
        result
    )


    if (
        (index + 1) % 25 == 0
        or
        index == total - 1
    ):

        print(
            f"Progress: "
            f"{index + 1:,}/{total:,}"
        )


CANDIDATE_EVALUATION_DF = pd.DataFrame(
    EVALUATION_RESULTS
)


print("\nEvaluation completed.")

print(
    "Results:",
    CANDIDATE_EVALUATION_DF.shape
)

In [ ]:
# ============================================================
# NOTEBOOK 06.22 — EVALUATION STATUS SUMMARY
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.22")
print("EVALUATION STATUS SUMMARY")
print("=" * 100)


status_summary = (
    CANDIDATE_EVALUATION_DF[
        "status"
    ]
    .value_counts(
        dropna=False
    )
)


display(
    status_summary.to_frame(
        "count"
    )
)


print(
    "\nSuccessful:",
    int(
        (
            CANDIDATE_EVALUATION_DF[
                "status"
            ]
            == "SUCCESS"
        ).sum()
    )
)

print(
    "Failed:",
    int(
        (
            CANDIDATE_EVALUATION_DF[
                "status"
            ]
            == "FAILED"
        ).sum()
    )
)

print(
    "Skipped:",
    int(
        (
            CANDIDATE_EVALUATION_DF[
                "status"
            ]
            == "SKIPPED"
        ).sum()
    )
)

In [ ]:
# ============================================================
# NOTEBOOK 06.23 — OBJECTIVE CANDIDATE RANKING
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.23")
print("OBJECTIVE CANDIDATE RANKING")
print("=" * 100)


SUCCESS_DF = (
    CANDIDATE_EVALUATION_DF[
        CANDIDATE_EVALUATION_DF[
            "status"
        ] == "SUCCESS"
    ]
    .copy()
)


def rank_numeric_group(
    group
):

    group = group.copy()

    group["primary_score"] = (
        group["nrmse"]
    )

    group["rank"] = (
        group[
            "primary_score"
        ]
        .rank(
            method="min",
            ascending=True
        )
    )

    return group


def rank_categorical_group(
    group
):

    group = group.copy()

    group["primary_score"] = (
        group["balanced_accuracy"]
    )

    group["secondary_score"] = (
        group["macro_f1"]
    )

    group = group.sort_values(
        [
            "primary_score",
            "secondary_score"
        ],
        ascending=[
            False,
            False
        ]
    )

    group["rank"] = np.arange(
        1,
        len(group) + 1
    )

    return group


RANKED_GROUPS = []


GROUP_COLUMNS = [
    "dataset_id",
    "feature",
    "missingness_mechanism",
    "missingness_rate"
]


for group_key, group in (
    SUCCESS_DF.groupby(
        GROUP_COLUMNS,
        dropna=False
    )
):

    if (
        group["feature_type"].iloc[0]
        == "numeric"
    ):

        ranked = rank_numeric_group(
            group
        )

    else:

        ranked = rank_categorical_group(
            group
        )


    RANKED_GROUPS.append(
        ranked
    )


RANKED_EVALUATION_DF = pd.concat(
    RANKED_GROUPS,
    ignore_index=True
)


print(
    "Ranking completed."
)

display(
    RANKED_EVALUATION_DF.head(20)
)

In [ ]:
# ============================================================
# NOTEBOOK 06.24 — AGGREGATE STRATEGY PERFORMANCE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.24")
print("AGGREGATE STRATEGY PERFORMANCE")
print("=" * 100)


NUMERIC_RESULTS = SUCCESS_DF[
    SUCCESS_DF[
        "feature_type"
    ] == "numeric"
]


CATEGORICAL_RESULTS = SUCCESS_DF[
    SUCCESS_DF[
        "feature_type"
    ] == "categorical"
]


NUMERIC_STRATEGY_SUMMARY = (
    NUMERIC_RESULTS
    .groupby(
        "strategy_id"
    )
    .agg(
        mean_rmse=(
            "rmse",
            "mean"
        ),
        mean_mae=(
            "mae",
            "mean"
        ),
        mean_nrmse=(
            "nrmse",
            "mean"
        ),
        mean_r2=(
            "r2",
            "mean"
        ),
        mean_runtime_seconds=(
            "runtime_seconds",
            "mean"
        ),
        evaluations=(
            "strategy_id",
            "size"
        )
    )
    .reset_index()
)


CATEGORICAL_STRATEGY_SUMMARY = (
    CATEGORICAL_RESULTS
    .groupby(
        "strategy_id"
    )
    .agg(
        mean_accuracy=(
            "accuracy",
            "mean"
        ),
        mean_balanced_accuracy=(
            "balanced_accuracy",
            "mean"
        ),
        mean_macro_f1=(
            "macro_f1",
            "mean"
        ),
        mean_runtime_seconds=(
            "runtime_seconds",
            "mean"
        ),
        evaluations=(
            "strategy_id",
            "size"
        )
    )
    .reset_index()
)


print("\nNUMERIC STRATEGIES")
display(
    NUMERIC_STRATEGY_SUMMARY
)


print("\nCATEGORICAL STRATEGIES")
display(
    CATEGORICAL_STRATEGY_SUMMARY
)

In [ ]:
# ============================================================
# NOTEBOOK 06.25 — LLM VS NON-LLM COMPARISON
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.25")
print("LLM-RECOMMENDED VS NON-LLM CANDIDATES")
print("=" * 100)


COMPARISON_ROWS = []


for group_key, group in (
    SUCCESS_DF.groupby(
        [
            "dataset_id",
            "feature",
            "missingness_mechanism",
            "missingness_rate"
        ]
    )
):

    feature_type = (
        group["feature_type"].iloc[0]
    )


    if feature_type == "numeric":

        metric = "nrmse"

        best_is_lower = True

    else:

        metric = "balanced_accuracy"

        best_is_lower = False


    for source_name, source_df in [
        (
            "LLM_RECOMMENDED",
            group[
                group[
                    "llm_recommended"
                ]
            ]
        ),
        (
            "NON_LLM",
            group[
                ~group[
                    "llm_recommended"
                ]
            ]
        )
    ]:

        if source_df.empty:

            continue


        values = source_df[
            metric
        ].dropna()


        if values.empty:

            continue


        if best_is_lower:

            best_value = values.min()

        else:

            best_value = values.max()


        COMPARISON_ROWS.append({

            "dataset_id":
                group_key[0],

            "feature":
                group_key[1],

            "missingness_mechanism":
                group_key[2],

            "missingness_rate":
                group_key[3],

            "feature_type":
                feature_type,

            "source":
                source_name,

            "metric":
                metric,

            "best_value":
                float(best_value)

        })


LLM_COMPARISON_DF = pd.DataFrame(
    COMPARISON_ROWS
)


display(
    LLM_COMPARISON_DF.head(20)
)

print(
    "\nLLM comparison : COMPLETED"
)

In [ ]:
# ============================================================
# NOTEBOOK 06.26 — LLM RECOMMENDATION WIN RATE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.26")
print("LLM RECOMMENDATION EMPIRICAL WIN RATE")
print("=" * 100)


WIN_ROWS = []


for group_key, group in (
    SUCCESS_DF.groupby(
        [
            "dataset_id",
            "feature",
            "missingness_mechanism",
            "missingness_rate"
        ]
    )
):

    llm_group = group[
        group[
            "llm_recommended"
        ]
    ]

    if llm_group.empty:

        continue


    feature_type = (
        group["feature_type"].iloc[0]
    )


    if feature_type == "numeric":

        metric = "nrmse"

        best_strategy = (
            group.loc[
                group[metric].idxmin(),
                "strategy_id"
            ]
        )

    else:

        metric = "balanced_accuracy"

        best_strategy = (
            group.loc[
                group[metric].idxmax(),
                "strategy_id"
            ]
        )


    llm_wins = (
        best_strategy
        in
        llm_group[
            "strategy_id"
        ].tolist()
    )


    WIN_ROWS.append({

        "dataset_id":
            group_key[0],

        "feature":
            group_key[1],

        "missingness_mechanism":
            group_key[2],

        "missingness_rate":
            group_key[3],

        "feature_type":
            feature_type,

        "best_strategy":
            best_strategy,

        "llm_recommendation_wins":
            bool(llm_wins)

    })


LLM_WIN_DF = pd.DataFrame(
    WIN_ROWS
)


if not LLM_WIN_DF.empty:

    win_rate = (
        LLM_WIN_DF[
            "llm_recommendation_wins"
        ]
        .mean()
    )

    print(
        f"LLM empirical win rate: "
        f"{win_rate:.3f}"
    )

else:

    win_rate = np.nan

    print(
        "LLM win rate unavailable."
    )


display(
    LLM_WIN_DF.head(20)
)

In [ ]:
# ============================================================
# NOTEBOOK 06.27 — MISSINGNESS ROBUSTNESS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.27")
print("ROBUSTNESS ACROSS MISSINGNESS LEVELS")
print("=" * 100)


ROBUSTNESS_ROWS = []


for strategy_id, group in (
    SUCCESS_DF.groupby(
        "strategy_id"
    )
):

    feature_type = (
        group["feature_type"]
        .mode()
        .iloc[0]
    )


    if feature_type == "numeric":

        metric = "nrmse"

    else:

        metric = "balanced_accuracy"


    for missing_rate in (
        MISSINGNESS_LEVELS
    ):

        subset = group[
            np.isclose(
                group[
                    "missingness_rate"
                ],
                missing_rate
            )
        ]


        if subset.empty:

            continue


        ROBUSTNESS_ROWS.append({

            "strategy_id":
                strategy_id,

            "feature_type":
                feature_type,

            "missingness_rate":
                missing_rate,

            "mean_metric":
                float(
                    subset[
                        metric
                    ].mean()
                ),

            "metric":
                metric

        })


ROBUSTNESS_DF = pd.DataFrame(
    ROBUSTNESS_ROWS
)


display(
    ROBUSTNESS_DF
)

In [ ]:
# ============================================================
# NOTEBOOK 06.28 — STRATEGY STABILITY SCORE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.28")
print("STRATEGY STABILITY SCORE")
print("=" * 100)


STABILITY_ROWS = []


for strategy_id, group in (
    SUCCESS_DF.groupby(
        "strategy_id"
    )
):

    numeric_group = group[
        group[
            "feature_type"
        ] == "numeric"
    ]

    categorical_group = group[
        group[
            "feature_type"
        ] == "categorical"
    ]


    if not numeric_group.empty:

        values = (
            numeric_group[
                "nrmse"
            ]
            .dropna()
        )

        if len(values) > 1:

            cv = (
                values.std()
                /
                max(
                    values.mean(),
                    1e-12
                )
            )

            stability = (
                1.0
                /
                (1.0 + cv)
            )

            STABILITY_ROWS.append({

                "strategy_id":
                    strategy_id,

                "feature_type":
                    "numeric",

                "coefficient_variation":
                    float(cv),

                "stability_score":
                    float(stability)

            })


    if not categorical_group.empty:

        values = (
            categorical_group[
                "balanced_accuracy"
            ]
            .dropna()
        )

        if len(values) > 1:

            cv = (
                values.std()
                /
                max(
                    abs(values.mean()),
                    1e-12
                )
            )

            stability = (
                1.0
                /
                (1.0 + cv)
            )

            STABILITY_ROWS.append({

                "strategy_id":
                    strategy_id,

                "feature_type":
                    "categorical",

                "coefficient_variation":
                    float(cv),

                "stability_score":
                    float(stability)

            })


STRATEGY_STABILITY_DF = pd.DataFrame(
    STABILITY_ROWS
)


display(
    STRATEGY_STABILITY_DF
)

In [ ]:
# ============================================================
# NOTEBOOK 06.29 — EMPIRICAL STRATEGY SCORE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.29")
print("EMPIRICAL STRATEGY SCORE")
print("=" * 100)


def minmax_normalize(
    series,
    inverse=False
):

    series = pd.Series(
        series,
        dtype=float
    )

    minimum = series.min()
    maximum = series.max()

    if (
        pd.isna(minimum)
        or
        pd.isna(maximum)
        or
        maximum == minimum
    ):

        result = pd.Series(
            1.0,
            index=series.index
        )

    else:

        result = (
            (series - minimum)
            /
            (maximum - minimum)
        )


    if inverse:

        result = 1.0 - result


    return result


SCORE_ROWS = []


for strategy_id, group in (
    SUCCESS_DF.groupby(
        "strategy_id"
    )
):

    feature_type = (
        group["feature_type"]
        .mode()
        .iloc[0]
    )


    if feature_type == "numeric":

        metric_values = (
            group["nrmse"]
            .dropna()
        )

        performance = (
            1.0
            -
            minmax_normalize(
                metric_values,
                inverse=False
            ).mean()
        )

    else:

        metric_values = (
            group[
                "balanced_accuracy"
            ]
            .dropna()
        )

        performance = (
            minmax_normalize(
                metric_values,
                inverse=False
            ).mean()
        )


    runtime = (
        group[
            "runtime_seconds"
        ]
        .mean()
    )


    stability_match = (
        STRATEGY_STABILITY_DF[
            (
                STRATEGY_STABILITY_DF[
                    "strategy_id"
                ]
                == strategy_id
            )
            &
            (
                STRATEGY_STABILITY_DF[
                    "feature_type"
                ]
                == feature_type
            )
        ]
    )


    if not stability_match.empty:

        stability = float(
            stability_match[
                "stability_score"
            ].iloc[0]
        )

    else:

        stability = 0.5


    final_score = (
        0.70 * performance
        +
        0.20 * stability
        +
        0.10 * (
            1.0
            /
            (1.0 + runtime)
        )
    )


    SCORE_ROWS.append({

        "strategy_id":
            strategy_id,

        "feature_type":
            feature_type,

        "performance_score":
            float(performance),

        "stability_score":
            float(stability),

        "mean_runtime_seconds":
            float(runtime),

        "empirical_score":
            float(final_score)

    })


EMPIRICAL_STRATEGY_SCORE_DF = (
    pd.DataFrame(
        SCORE_ROWS
    )
    .sort_values(
        "empirical_score",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


display(
    EMPIRICAL_STRATEGY_SCORE_DF
)

In [ ]:
# ============================================================
# NOTEBOOK 06.30 — FEATURE-LEVEL CANDIDATE RANKING
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.30")
print("FEATURE-LEVEL CANDIDATE RANKING")
print("=" * 100)


FEATURE_RANK_ROWS = []


for group_key, group in (
    SUCCESS_DF.groupby(
        [
            "dataset_id",
            "feature",
            "missingness_mechanism",
            "missingness_rate"
        ]
    )
):

    feature_type = (
        group["feature_type"].iloc[0]
    )


    if feature_type == "numeric":

        metric = "nrmse"

        sorted_group = group.sort_values(
            metric,
            ascending=True
        )

    else:

        metric = "balanced_accuracy"

        sorted_group = group.sort_values(
            metric,
            ascending=False
        )


    for rank, (_, row) in enumerate(
        sorted_group.iterrows(),
        start=1
    ):

        FEATURE_RANK_ROWS.append({

            "dataset_id":
                group_key[0],

            "feature":
                group_key[1],

            "missingness_mechanism":
                group_key[2],

            "missingness_rate":
                group_key[3],

            "feature_type":
                feature_type,

            "strategy_id":
                row[
                    "strategy_id"
                ],

            "rank":
                rank,

            "primary_metric":
                metric,

            "primary_metric_value":
                row[
                    metric
                ],

            "llm_recommended":
                row[
                    "llm_recommended"
                ]

        })


FEATURE_CANDIDATE_RANKING_DF = pd.DataFrame(
    FEATURE_RANK_ROWS
)


display(
    FEATURE_CANDIDATE_RANKING_DF.head(30)
)

In [ ]:
# ============================================================
# NOTEBOOK 06.31 — FINAL EMPIRICAL WINNER
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.31")
print("FINAL EMPIRICAL WINNER PER FEATURE")
print("=" * 100)


FINAL_WINNER_DF = (
    FEATURE_CANDIDATE_RANKING_DF[
        FEATURE_CANDIDATE_RANKING_DF[
            "rank"
        ] == 1
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


print(
    f"Final winner records: "
    f"{len(FINAL_WINNER_DF):,}"
)

display(
    FINAL_WINNER_DF.head(30)
)

In [ ]:
# ============================================================
# NOTEBOOK 06.32 — LLM / EMPIRICAL AGREEMENT
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.32")
print("LLM / EMPIRICAL AGREEMENT")
print("=" * 100)


AGREEMENT_ROWS = []


for group_key, group in (
    SUCCESS_DF.groupby(
        [
            "dataset_id",
            "feature",
            "missingness_mechanism",
            "missingness_rate"
        ]
    )
):

    feature_type = (
        group["feature_type"].iloc[0]
    )


    if feature_type == "numeric":

        best_strategy = (
            group.loc[
                group[
                    "nrmse"
                ].idxmin(),
                "strategy_id"
            ]
        )

    else:

        best_strategy = (
            group.loc[
                group[
                    "balanced_accuracy"
                ].idxmax(),
                "strategy_id"
            ]
        )


    llm_strategies = (
        group[
            group[
                "llm_recommended"
            ]
        ][
            "strategy_id"
        ]
        .tolist()
    )


    agreement = (
        best_strategy
        in
        llm_strategies
    )


    AGREEMENT_ROWS.append({

        "dataset_id":
            group_key[0],

        "feature":
            group_key[1],

        "missingness_mechanism":
            group_key[2],

        "missingness_rate":
            group_key[3],

        "feature_type":
            feature_type,

        "empirical_best_strategy":
            best_strategy,

        "llm_recommended_strategies":
            "|".join(
                llm_strategies
            ),

        "agreement":
            bool(agreement)

    })


LLM_EMPIRICAL_AGREEMENT_DF = (
    pd.DataFrame(
        AGREEMENT_ROWS
    )
)


if not LLM_EMPIRICAL_AGREEMENT_DF.empty:

    agreement_rate = (
        LLM_EMPIRICAL_AGREEMENT_DF[
            "agreement"
        ]
        .mean()
    )

    print(
        f"Overall agreement: "
        f"{agreement_rate:.3f}"
    )


display(
    LLM_EMPIRICAL_AGREEMENT_DF.head(30)
)

In [ ]:
# ============================================================
# NOTEBOOK 06.33 — CANDIDATE EVIDENCE TABLE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.33")
print("CANDIDATE EVIDENCE TABLE")
print("=" * 100)


CANDIDATE_EVIDENCE_ROWS = []


for group_key, group in (
    SUCCESS_DF.groupby(
        [
            "dataset_id",
            "feature",
            "missingness_mechanism",
            "missingness_rate"
        ]
    )
):

    feature_type = (
        group["feature_type"].iloc[0]
    )


    if feature_type == "numeric":

        group = group.sort_values(
            "nrmse",
            ascending=True
        )

    else:

        group = group.sort_values(
            "balanced_accuracy",
            ascending=False
        )


    for rank, (_, row) in enumerate(
        group.iterrows(),
        start=1
    ):

        if feature_type == "numeric":

            evidence_metric = (
                row["nrmse"]
            )

            metric_name = "NRMSE"

        else:

            evidence_metric = (
                row[
                    "balanced_accuracy"
                ]
            )

            metric_name = (
                "Balanced Accuracy"
            )


        CANDIDATE_EVIDENCE_ROWS.append({

            "dataset_id":
                group_key[0],

            "feature":
                group_key[1],

            "feature_type":
                feature_type,

            "missingness_mechanism":
                group_key[2],

            "missingness_rate":
                group_key[3],

            "strategy_id":
                row["strategy_id"],

            "llm_recommended":
                row[
                    "llm_recommended"
                ],

            "empirical_rank":
                rank,

            "metric_name":
                metric_name,

            "metric_value":
                evidence_metric,

            "rmse":
                row["rmse"],

            "mae":
                row["mae"],

            "nrmse":
                row["nrmse"],

            "r2":
                row["r2"],

            "accuracy":
                row["accuracy"],

            "balanced_accuracy":
                row[
                    "balanced_accuracy"
                ],

            "macro_f1":
                row["macro_f1"],

            "runtime_seconds":
                row[
                    "runtime_seconds"
                ]

        })


CANDIDATE_EVIDENCE_DF = (
    pd.DataFrame(
        CANDIDATE_EVIDENCE_ROWS
    )
)


display(
    CANDIDATE_EVIDENCE_DF.head(30)
)

In [ ]:
# ============================================================
# NOTEBOOK 06.34 — RESEARCH VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.34")
print("RESEARCH VALIDATION CHECKS")
print("=" * 100)


VALIDATION_ROWS = []


def add_validation(
    check,
    passed,
    value,
    expected
):

    VALIDATION_ROWS.append({

        "check":
            check,

        "passed":
            bool(passed),

        "value":
            str(value),

        "expected":
            str(expected)

    })


add_validation(
    "Evaluation results exist",
    len(
        CANDIDATE_EVALUATION_DF
    ) > 0,
    len(
        CANDIDATE_EVALUATION_DF
    ),
    "> 0"
)


add_validation(
    "Successful evaluations exist",
    len(
        SUCCESS_DF
    ) > 0,
    len(
        SUCCESS_DF
    ),
    "> 0"
)


add_validation(
    "All datasets represented",
    set(DATASETS).issubset(
        set(
            CANDIDATE_EVALUATION_DF[
                "dataset_id"
            ]
            .unique()
        )
    ),
    CANDIDATE_EVALUATION_DF[
        "dataset_id"
    ].nunique(),
    len(DATASETS)
)


add_validation(
    "All missingness mechanisms represented",
    set(
        MISSINGNESS_MECHANISMS
    ).issubset(
        set(
            SUCCESS_DF[
                "missingness_mechanism"
            ].unique()
        )
    ),
    SUCCESS_DF[
        "missingness_mechanism"
    ].unique(),
    MISSINGNESS_MECHANISMS
)


add_validation(
    "All configured missingness rates represented",
    set(
        MISSINGNESS_LEVELS
    ).issubset(
        set(
            SUCCESS_DF[
                "missingness_rate"
            ].unique()
        )
    ),
    SUCCESS_DF[
        "missingness_rate"
    ].unique(),
    MISSINGNESS_LEVELS
)


add_validation(
    "No negative RMSE",
    (
        SUCCESS_DF[
            "rmse"
        ]
        .dropna()
        >= 0
    ).all(),
    "checked",
    "all >= 0"
)


add_validation(
    "No negative MAE",
    (
        SUCCESS_DF[
            "mae"
        ]
        .dropna()
        >= 0
    ).all(),
    "checked",
    "all >= 0"
)


add_validation(
    "Classification accuracy bounded",
    (
        SUCCESS_DF[
            "accuracy"
        ]
        .dropna()
        .between(
            0,
            1
        )
    ).all(),
    "checked",
    "[0,1]"
)


add_validation(
    "Balanced accuracy bounded",
    (
        SUCCESS_DF[
            "balanced_accuracy"
        ]
        .dropna()
        .between(
            0,
            1
        )
    ).all(),
    "checked",
    "[0,1]"
)


VALIDATION_DF = pd.DataFrame(
    VALIDATION_ROWS
)


display(
    VALIDATION_DF
)


if not VALIDATION_DF[
    "passed"
].all():

    raise RuntimeError(
        "One or more research "
        "validation checks failed."
    )


print(
    "\nALL VALIDATION CHECKS : PASSED"
)

In [ ]:
# ============================================================
# NOTEBOOK 06.35 — SAVE CORE RESULTS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.35")
print("SAVE CORE EVALUATION RESULTS")
print("=" * 100)


OUTPUT_FILES = {}


def save_dataframe(
    df,
    filename
):

    path = RESULTS_DIR / filename

    df.to_csv(
        path,
        index=False
    )

    OUTPUT_FILES[
        filename
    ] = path

    print(
        f"Saved: {path}"
    )


save_dataframe(
    STRATEGY_REGISTRY_DF,
    "strategy_registry.csv"
)

save_dataframe(
    EVALUATION_PLAN_DF,
    "evaluation_plan.csv"
)

save_dataframe(
    CANDIDATE_EVALUATION_DF,
    "candidate_evaluation_results.csv"
)

save_dataframe(
    RANKED_EVALUATION_DF,
    "ranked_evaluation_results.csv"
)

save_dataframe(
    NUMERIC_STRATEGY_SUMMARY,
    "numeric_strategy_summary.csv"
)

save_dataframe(
    CATEGORICAL_STRATEGY_SUMMARY,
    "categorical_strategy_summary.csv"
)

save_dataframe(
    ROBUSTNESS_DF,
    "missingness_robustness.csv"
)

save_dataframe(
    STRATEGY_STABILITY_DF,
    "strategy_stability.csv"
)

save_dataframe(
    EMPIRICAL_STRATEGY_SCORE_DF,
    "empirical_strategy_scores.csv"
)

save_dataframe(
    FEATURE_CANDIDATE_RANKING_DF,
    "feature_candidate_ranking.csv"
)

save_dataframe(
    FINAL_WINNER_DF,
    "final_empirical_winners.csv"
)

save_dataframe(
    LLM_EMPIRICAL_AGREEMENT_DF,
    "llm_empirical_agreement.csv"
)

save_dataframe(
    CANDIDATE_EVIDENCE_DF,
    "candidate_evidence.csv"
)

save_dataframe(
    VALIDATION_DF,
    "candidate_evaluation_validation.csv"
)


print(
    f"\nSaved artifacts: "
    f"{len(OUTPUT_FILES)}"
)

In [ ]:
# ============================================================
# NOTEBOOK 06.36 — RESEARCH SUMMARY
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.36")
print("CANDIDATE EVALUATION RESEARCH SUMMARY")
print("=" * 100)


print("\nDATASETS")
print("-" * 100)

for dataset_id in DATASETS:

    if dataset_id in EVALUATION_DATA:

        print(
            f"{dataset_id:25s}: "
            f"{len(EVALUATION_DATA[dataset_id]):,} rows"
        )


print("\nEVALUATION")
print("-" * 100)

print(
    "Total configurations :",
    len(
        CANDIDATE_EVALUATION_DF
    )
)

print(
    "Successful           :",
    len(
        SUCCESS_DF
    )
)

print(
    "Missingness mechanisms:",
    MISSINGNESS_MECHANISMS
)

print(
    "Missingness levels   :",
    MISSINGNESS_LEVELS
)


print("\nLLM AGREEMENT")
print("-" * 100)

if not LLM_EMPIRICAL_AGREEMENT_DF.empty:

    print(
        f"Agreement rate: "
        f"{LLM_EMPIRICAL_AGREEMENT_DF['agreement'].mean():.3f}"
    )


print("\n" + "=" * 100)
print("CANDIDATE STRATEGY EVALUATION : COMPLETED")
print("=" * 100)

In [ ]:
# ============================================================
# NOTEBOOK 06.37 — FILE HASHING
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.37")
print("REPRODUCIBILITY HASHING")
print("=" * 100)


def sha256_file(
    path
):

    sha256 = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda:
                f.read(
                    1024 * 1024
                ),
            b""
        ):

            sha256.update(
                chunk
            )


    return sha256.hexdigest()


print(
    "SHA-256 utility : READY"
)

In [ ]:
# ============================================================
# NOTEBOOK 06.38 — OUTPUT MANIFEST
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.38")
print("OUTPUT MANIFEST")
print("=" * 100)


OUTPUT_MANIFEST = {}


for filename, path in OUTPUT_FILES.items():

    if path.exists():

        OUTPUT_MANIFEST[
            filename
        ] = {

            "exists":
                True,

            "size_bytes":
                path.stat().st_size,

            "sha256":
                sha256_file(
                    path
                )

        }

    else:

        OUTPUT_MANIFEST[
            filename
        ] = {

            "exists":
                False,

            "size_bytes":
                0,

            "sha256":
                None

        }


MANIFEST = {

    "project":
        "AIR-LLM",

    "notebook":
        "06_Candidate_Strategy_Evaluation",

    "version":
        "1.0",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "master_seed":
        MASTER_SEED,

    "datasets":
        DATASETS,

    "target_registry":
        TARGET_REGISTRY,

    "evaluation_sample_size":
        EVALUATION_SAMPLE_SIZE,

    "missingness_levels":
        MISSINGNESS_LEVELS,

    "missingness_mechanisms":
        MISSINGNESS_MECHANISMS,

    "knn_neighbors":
        KNN_NEIGHBORS,

    "tree_estimators":
        TREE_ESTIMATORS,

    "strategy_count":
        len(
            STRATEGY_REGISTRY
        ),

    "evaluation_rows":
        int(
            len(
                CANDIDATE_EVALUATION_DF
            )
        ),

    "successful_evaluations":
        int(
            len(
                SUCCESS_DF
            )
        ),

    "llm_agreement_rate":
        (
            float(
                LLM_EMPIRICAL_AGREEMENT_DF[
                    "agreement"
                ].mean()
            )
            if not
            LLM_EMPIRICAL_AGREEMENT_DF.empty
            else None
        ),

    "outputs":
        OUTPUT_MANIFEST
}


MANIFEST_PATH = (
    MANIFEST_DIR
    /
    "notebook_06_manifest.json"
)


with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        MANIFEST,
        f,
        indent=2
    )


print(
    f"Manifest saved:\n"
    f"{MANIFEST_PATH}"
)

print(
    f"Output artifacts: "
    f"{len(OUTPUT_MANIFEST)}"
)

In [ ]:
# ============================================================
# NOTEBOOK 06.39 — FINAL INTEGRITY CHECK
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.39")
print("FINAL NOTEBOOK INTEGRITY CHECK")
print("=" * 100)


required_objects = {

    "TRAINING_DATA":
        "TRAINING_DATA",

    "FEATURE_PROFILE_DF":
        "FEATURE_PROFILE_DF",

    "STRATEGY_REGISTRY":
        "STRATEGY_REGISTRY",

    "EVALUATION_PLAN_DF":
        "EVALUATION_PLAN_DF",

    "CANDIDATE_EVALUATION_DF":
        "CANDIDATE_EVALUATION_DF",

    "RANKED_EVALUATION_DF":
        "RANKED_EVALUATION_DF",

    "FINAL_WINNER_DF":
        "FINAL_WINNER_DF",

    "CANDIDATE_EVIDENCE_DF":
        "CANDIDATE_EVIDENCE_DF",

    "VALIDATION_DF":
        "VALIDATION_DF"

}


INTEGRITY_ROWS = []


for name in required_objects:

    exists = (
        name in globals()
    )

    INTEGRITY_ROWS.append({

        "object":
            name,

        "exists":
            exists

    })


INTEGRITY_DF = pd.DataFrame(
    INTEGRITY_ROWS
)


display(
    INTEGRITY_DF
)


if not INTEGRITY_DF[
    "exists"
].all():

    missing = (
        INTEGRITY_DF[
            ~INTEGRITY_DF[
                "exists"
            ]
        ]["object"]
        .tolist()
    )

    raise RuntimeError(
        "Missing required objects: "
        + ", ".join(missing)
    )


print("\n" + "=" * 100)
print("NOTEBOOK 06 INTEGRITY : PASSED")
print("=" * 100)


gc.collect()

if "torch" in globals():

    if torch.cuda.is_available():

        torch.cuda.empty_cache()